In [2]:
import pandas as pd
import numpy as np

# Ejemplo de datos
data = {
    "HomeTeam": ["TeamA", "TeamB", "TeamC", "TeamA", "TeamB"],
    "AwayTeam": ["TeamB", "TeamC", "TeamA", "TeamC", "TeamA"],
    "HomeGoals": [2, 1, 3, 0, 1],
    "AwayGoals": [1, 0, 2, 2, 3],
}

# Crear el DataFrame
results = pd.DataFrame(data)

# Calcular goles totales anotados y recibidos por cada equipo en casa y como visitante
home_stats = results.groupby("HomeTeam").agg(
    HomeGoalsScored=("HomeGoals", "sum"), HomeGames=("HomeGoals", "count")
).reset_index()

away_stats = results.groupby("AwayTeam").agg(
    AwayGoalsScored=("AwayGoals", "sum"), AwayGames=("AwayGoals", "count")
).reset_index()

# Calcular el promedio de goles anotados por partido
home_stats["AvgHomeGoals"] = home_stats["HomeGoalsScored"] / home_stats["HomeGames"]
away_stats["AvgAwayGoals"] = away_stats["AwayGoalsScored"] / away_stats["AwayGames"]

# Calcular los goles recibidos por cada equipo
home_conceded = results.groupby("HomeTeam").agg(HomeGoalsConceded=("AwayGoals", "sum"))
away_conceded = results.groupby("AwayTeam").agg(AwayGoalsConceded=("HomeGoals", "sum"))

# Combinar datos de promedio de goles anotados y recibidos
team_stats = home_stats.merge(home_conceded, on="HomeTeam").merge(
    away_stats.merge(away_conceded, on="AwayTeam"), left_on="HomeTeam", right_on="AwayTeam"
)

# Renombrar columnas para claridad
team_stats = team_stats.rename(columns={"HomeTeam": "Team"}).drop(columns="AwayTeam")

# Crear la tabla de índices de Poisson
team_stats["PoissonHome"] = team_stats["AvgHomeGoals"]
team_stats["PoissonAway"] = team_stats["AvgAwayGoals"]

# Mostrar la tabla final
team_stats

,Team,HomeGoalsScored,HomeGames,AvgHomeGoals,HomeGoalsConceded,AwayGoalsScored,AwayGames,AvgAwayGoals,AwayGoalsConceded,PoissonHome,PoissonAway
0,TeamA,2,2,1.0,3,5,2,2.5,4,1.0,2.5
1,TeamB,2,2,1.0,3,1,1,1.0,2,1.0,1.0
2,TeamC,3,1,3.0,2,2,2,1.0,1,3.0,1.0
